In [1]:
import sys

dir = '../..'
if dir not in sys.path:
    sys.path.append(dir)

In [2]:
from immGen import *

### Verify the SelectType Module
we will check that is able to recognize all instruction types

In [3]:
hw = py4hw.HWSystem()

opcode = hw.wire('opcode', 7)
imm_type = hw.wire('imm_type', 3)

dut = SelectType(hw, 'selectType', opcode, imm_type)

In [4]:
import punxa
from punxa.assembly import assemble

In [5]:
test_opcode = assemble('addi x10, x11, 1') & ((1<<7)-1)
opcode.put(test_opcode)
hw.getSimulator().clk()
print('Type:', imm_type.get())


Type: 0


In [6]:
def check_type(ins, expected):
    test_opcode = assemble(ins) & ((1<<7)-1)
    opcode.put(test_opcode)
    hw.getSimulator().clk()
    val = imm_type.get()
    val_map = ['I', 'S', 'B', 'U', 'J']
    print('Instruction:', ins, '\tType:', val_map[val], '\tExpected:', val_map[expected], '\t[OK]' if (val == expected) else '\t[ERROR]')

In [7]:
check_type('addi x10, x11, 1', 0)
check_type('sw x10, 8(x15)', 1)
check_type('beq x5, x6, 2', 2)
check_type('lui x10, 0x12345', 3)
check_type('jal x0, 55', 4)

Instruction: addi x10, x11, 1 	Type: I 	Expected: I 	[OK]
Instruction: sw x10, 8(x15) 	Type: S 	Expected: S 	[OK]
Instruction: beq x5, x6, 2 	Type: B 	Expected: B 	[OK]
Instruction: lui x10, 0x12345 	Type: U 	Expected: U 	[OK]
Instruction: jal x0, 55 	Type: J 	Expected: J 	[OK]


### Verify the immGen Module 

In [8]:
hw = py4hw.HWSystem()

ir = hw.wire('ir', 32)
imm = hw.wire('imm', 32)

dut = immGen(hw, 'selectType', ir, imm)

In [9]:
def check_imm(ins, expected):
    test_ins = assemble(ins) 
    ir.put(test_ins)
    hw.getSimulator().clk()
    val = py4hw.IntegerHelper.c2_to_signed(imm.get(), 32)
    
    print('Instruction:', ins, '\tImm:', val, '\tExpected:', expected, '\t[OK]' if (val == expected) else '\t[ERROR]')

In [10]:
check_imm('addi x10, x11, 10', 10)
check_imm('addi x10, x11, -1', -1)
check_imm('sw x10, 80(x15)', 80)
check_imm('sw x10, -8(x15)', -8)
check_imm('beq x5, x6, 20', 20)
check_imm('beq x5, x6, -20', -20)
check_imm('lui x10, 45', 45 << 12)
check_imm('jal x0, 50', 50)
check_imm('jal x0, -50', -50)

Instruction: addi x10, x11, 10 	Imm: 10 	Expected: 10 	[OK]
Instruction: addi x10, x11, -1 	Imm: -1 	Expected: -1 	[OK]
Instruction: sw x10, 80(x15) 	Imm: 80 	Expected: 80 	[OK]
Instruction: sw x10, -8(x15) 	Imm: -8 	Expected: -8 	[OK]
Instruction: beq x5, x6, 20 	Imm: 20 	Expected: 20 	[OK]
Instruction: beq x5, x6, -20 	Imm: -20 	Expected: -20 	[OK]
Instruction: lui x10, 45 	Imm: 184320 	Expected: 184320 	[OK]
Instruction: jal x0, 50 	Imm: 50 	Expected: 50 	[OK]
Instruction: jal x0, -50 	Imm: -50 	Expected: -50 	[OK]


In [11]:
rtl = py4hw.VerilogGenerator(dut)
verilog = rtl.getVerilogForHierarchy(dut)

print(verilog)

transpiling combinational /HWSystem[HWSystem]/immGen[selectType]/SelectType[imm_type]
// This file was automatically created by py4hw Verilog generator
module immGen (
	input [31:0] ir,
	output [31:0] imm);
wire [19:0] w_i4;
wire [31:0] w_imm_S;
wire [11:0] w_i3;
wire [31:0] w_imm_J;
wire w_i0;
wire w_i8;
wire [7:0] w_i9;
wire [6:0] w_opcode;
wire [3:0] w_i10;
wire [2:0] w_imm_typ;
wire [5:0] w_i6;
wire [4:0] w_i11;
wire [6:0] w_i5;
wire w_i12;
wire [11:0] w_i13;
wire [31:0] w_imm_B;
wire [12:0] w_i14;
wire w_i2;
wire [31:0] w_zero;
wire [9:0] w_i7;
wire [31:0] w_imm_I;
wire [31:0] w_imm_U;
wire [20:0] w_i15;
wire [11:0] w_i1;

assign w_opcode = ir[6:0];
SelectType_1c3535ed950 i_imm_type(.opcode(w_opcode),.imm_type(w_imm_typ));
assign w_i0 = 0;
assign w_i1[11:0] = 0;
assign w_i2 = ir[31];
assign w_i3 = ir[31:20];
assign w_i4 = ir[31:12];
assign w_i5 = ir[31:25];
assign w_i6 = ir[30:25];
assign w_i7 = ir[30:21];
assign w_i8 = ir[20];
assign w_i9 = ir[19:12];
assign w_i10 = ir[11:8];
ass